In [2]:
import json

import requests
import POKEMON.enums as enums
mainurl = "https://pokeapi.co/api/v2/"


In [12]:

url = mainurl + "pokemon-species/3"
response = requests.get(url)
print(response.json().keys())


dict_keys(['base_happiness', 'capture_rate', 'color', 'egg_groups', 'evolution_chain', 'evolves_from_species', 'flavor_text_entries', 'form_descriptions', 'forms_switchable', 'gender_rate', 'genera', 'generation', 'growth_rate', 'habitat', 'has_gender_differences', 'hatch_counter', 'id', 'is_baby', 'is_legendary', 'is_mythical', 'name', 'names', 'order', 'pal_park_encounters', 'pokedex_numbers', 'shape', 'varieties'])


In [ ]:
url = mainurl + "ability/1"
response = requests.get(url)
print(response.json()["name"].keys())

In [16]:
# get entried pokemon
filename = "JSON/pokeindex.json"
url = mainurl + "pokedex/36"  # champions dex
response = requests.get(url)
entry_numbers = []
for pokemon in response.json()["pokemon_entries"]:
    entry_numbers.append(int(pokemon["entry_number"]))
with open(filename, "w", encoding="utf-8") as f:
    f.write(json.dumps(entry_numbers, indent=4))


In [ ]:
# add megapokemon
filename = "JSON/pokeindex.json"
with open(filename, "r", encoding="utf-8") as f:
    pokeid = json.load(f)
addid = []
for id in pokeid:
    id = int(id)
    url = mainurl + "pokemon-species/" + str(id)
    response = requests.get(url)
    try:
        varieties = response.json()["varieties"]
        for variety in varieties:
            if variety["is_default"] is False:
                url = variety["pokemon"]["url"]
                response = requests.get(url)
                name = response.json()["name"]
                if "mega" in name:
                    print(f"mega exists: {name}")
                    addid.append(int(response.json()["id"]))
    except KeyError:
        print(f"スキップ：ID {id}")
pokeid += addid
with open(filename, "w", encoding="utf-8")as f:
    f.write(json.dumps(pokeid))

In [ ]:
import json

filename = "JSON/pokemon_data.json"

with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

# 1. まず全ポケモンにデフォルトで "mega_exsist": False を追加
for pokemon in data.values():
    pokemon["mega_exsist"] = False

# 2. 名前からデータを探せる参照用マップを作成
name_map = {pokemon["name"]: pokemon for pokemon in data.values()}

# 3. "-mega" を含むデータから元ポケモンを特定し、"mega_exsist": True に更新
for pokemon in list(data.values()):
    if "-mega" in pokemon["name"]:
        # "リザードン-mega-x" や "フシギバナ-mega" から元ポケモンの名前を取得
        base_name = pokemon["name"].split("-mega")[0]
        
        # 元のポケモンが存在すれば mega フラグを True に上書き
        if base_name in name_map:
            name_map[base_name]["mega_exsist"] = True

# 4. 更新結果をファイルに保存
with open("JSON/pokemon_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

In [9]:
# make pokemon.json

filename = "JSON/pokeindex.json"
with open(filename, "r", encoding="utf-8") as f:
    pokeid = json.load(f)
all_pokemon = {}
for id in pokeid:
    # get info
    id = int(id)
    url = mainurl + "pokemon/" + str(id)
    response = requests.get(url)
    # make case
    abilities = []
    moveids = []
    stats = dict()
    name = response.json()["name"]
    weight = response.json()["weight"]
    height = response.json()["height"]
    for ability in response.json()["abilities"]:
        dc = {}
        dc["ability_id"] = int(ability["ability"]["url"].split("/")[-2])
        dc["ability_name"] = ability["ability"]["name"]
        dc["is_hidden"] = ability["is_hidden"]
        abilities.append(dc)
    for move in response.json()["moves"]:
        version_group_details = move["version_group_details"]
        for version_group_detail in version_group_details:
            if version_group_detail["version_group"]["name"] == "champions":
                moveids.append(int(move["move"]["url"].split("/")[-2]))
    for stat in response.json()["stats"]:
        stat_id = int(stat["stat"]["url"].split("/")[-2])
        stat_value = stat["base_stat"]
        stats[stat_id] = stat_value
    types = []
    for type in response.json()["types"]:
        types.append(int(type["type"]["url"].split("/")[-2]))
    # add gender and jpname

    url = f"{mainurl}pokemon-species/{id}"
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()  # 404エラーなどを検知して例外（except）へ飛ばす
        data = response.json()
        # 性別比率の取得
        gender = data.get("gender_rate")
        jpname = next(
                (
                    name["name"]
                    for name in data.get("names", [])
                    if name["language"]["name"] in ("ja-hrkt", "ja")
                ),
                None,  # 見つからなかった場合のデフォルト値
            )
    except (requests.RequestException, KeyError, ValueError) as e:
        print(f"ID {id} のデータ取得に失敗しました: {e}")
        gender = None
        jpname = None
    all_pokemon[id] = {
        "name": name,
        "jpname": jpname,
        "types": types,
        "abilities": abilities,
        "moveids": moveids,
        "stats": stats,
        "gender": gender,
        "weight": weight,
        "height": height
    }
    print(f"{jpname}が終了しました。")
filename = "JSON/pokemon_data.json"
with open(filename, "w", encoding="utf-8") as f:
    f.write(json.dumps(all_pokemon, indent=4, ensure_ascii=False))

フシギバナが終了しました。
リザードンが終了しました。
カメックスが終了しました。
スピアーが終了しました。
ピジョットが終了しました。
アーボックが終了しました。
ピカチュウが終了しました。
ライチュウが終了しました。
ピクシーが終了しました。
キュウコンが終了しました。
ラフレシアが終了しました。
ウインディが終了しました。
フーディンが終了しました。
カイリキーが終了しました。
ウツボットが終了しました。
ヤドランが終了しました。
ゲンガーが終了しました。
ガルーラが終了しました。
スターミーが終了しました。
カイロスが終了しました。
ケンタロスが終了しました。
ギャラドスが終了しました。
メタモンが終了しました。
シャワーズが終了しました。
サンダースが終了しました。
ブースターが終了しました。
プテラが終了しました。
カビゴンが終了しました。
カイリューが終了しました。
メガニウムが終了しました。
バクフーンが終了しました。
オーダイルが終了しました。
アリアドスが終了しました。
デンリュウが終了しました。
マリルリが終了しました。
ニョロトノが終了しました。
エーフィが終了しました。
ブラッキーが終了しました。
ヤドキングが終了しました。
フォレトスが終了しました。
ハガネールが終了しました。
ハリーセンが終了しました。
ハッサムが終了しました。
ヘラクロスが終了しました。
エアームドが終了しました。
ヘルガーが終了しました。
バンギラスが終了しました。
ジュカインが終了しました。
バシャーモが終了しました。
ラグラージが終了しました。
ペリッパーが終了しました。
サーナイトが終了しました。
ヤミラミが終了しました。
クチートが終了しました。
ボスゴドラが終了しました。
チャーレムが終了しました。
ライボルトが終了しました。
サメハダーが終了しました。
バクーダが終了しました。
コータスが終了しました。
チルタリスが終了しました。
ミロカロスが終了しました。
ポワルンが終了しました。
ジュペッタが終了しました。
チリーンが終了しました。
アブソルが終了しました。
オニゴーリが終了しました。
メタグロスが終了しました。
ドダイトスが終了しました。
ゴウカザルが終了しました。
エンペルトが終了しました。
ムクホークが終了しました。
レントラーが終了しました。

In [7]:
import json

filename = "JSON/pokemon.json"

# 1. いったんファイルを読み込む
with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

# 2. ensure_ascii=False を指定して上書き保存する
with open(filename, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("日本語の復元（変換）が完了しました！")

日本語の復元（変換）が完了しました！


In [ ]:
# adding gender and jpname
filename = "JSON/pokemon_data.json"
with open(filename, "r", encoding="utf-8") as f:
    all_pokemon = json.load(f)
for id in all_pokemon:
    url = mainurl + "pokemon-species/" + str(id)
    response = requests.get(url)
    gender = response.json()["gender_rate"]
    jpname = response.json()["names"][0]["name"]
    all_pokemon[id]["gender"] = gender
    all_pokemon[id]["jpname"] = jpname
    print(all_pokemon[id]["jpname"])
    
json_string = json.dumps(all_pokemon, ensure_ascii=False)
with open(filename, "w", encoding="utf-8") as f:
    f.write(json_string)

In [ ]:
filename = "JSON/pokemon.json"
with open(filename, "r", encoding="utf-8") as f:
    poke = json.load(f)
all_pokemon = {}
url = mainurl + "pokemon/" + str(poke)

In [32]:

# making move.json
filename = "JSON/pokemon.json"
with open(filename, "r", encoding="utf-8") as f:
    pokemon_dict = json.load(f)
filename = "JSON/move.json"
with open(filename, "r", encoding="utf-8") as f:
    try:
        move_dict = json.load(f)
    except json.JSONDecodeError:
        # 万が一ファイルが空っぽなどで壊れていた場合の保険
        move_dict = dict()
for id in pokemon_dict:
    for moveid in list(pokemon_dict[id]["moveids"]):
        if str(moveid) not in move_dict:
            url = mainurl + "move/" + str(moveid)
            response = requests.get(url)
            effect_entrys = response.json()["effect_entries"]
            meta = response.json()["meta"]

            for effect_entry in effect_entrys:
                if effect_entry["language"]["name"] == "en":
                    short_effect = effect_entry["short_effect"]
                    break
            stat_changes = response.json()["stat_changes"]
            if len(stat_changes) == 0:
                stat_changes_value = 0
                stat_changes_stat = ""
            else:
                stat_changes_value = stat_changes[0]["change"]
                stat_changes_stat = stat_changes[0]["stat"]["name"]
            names = response.json()["names"]
            for name in names:
                if name["language"]["name"] == "ja-hrkt" or name["language"]["name"] == "ja":
                    jpname = name["name"]
                    break
                else:
                    print(f"{response.json()['name']} name was not found")
                    jpname = ""
            if type(meta) is dict:
                move_dict[str(moveid)] = {
                    "id": moveid,
                    "name": response.json()["name"],
                    "type": response.json()["type"]["name"],
                    "power": response.json()["power"],
                    "accuracy": response.json()["accuracy"],
                    "pp": response.json()["pp"],
                    "damage_class": response.json()["damage_class"]["name"],
                    "effect": short_effect,
                    "effect_chance": response.json()["effect_chance"],
                    "stat_changes_value": stat_changes_value,
                    "stat_changes_stat": stat_changes_stat,
                    "jpname": jpname,
                    # meta
                    "ailment": meta["ailment"]["name"],
                    "ailment_chance": meta["ailment_chance"],
                    "category": meta["category"]["name"],
                    "crit_rate": meta["crit_rate"],
                    "flinch_chance": meta["flinch_chance"],
                    "healing": meta["healing"],
                    "min_hits": meta["min_hits"],
                    "max_hits": meta["max_hits"],
                    "min_turns": meta["min_turns"],
                    "max_turns": meta["max_turns"],
                    "stat_chance": meta["stat_chance"],
                    "drain": meta["drain"],
                }
                print(f"追加：ID {moveid} - {move_dict[str(moveid)]['name']} exist meta")
            else:
                move_dict[str(moveid)] = {
                    "id": moveid,
                    "name": response.json()["name"],
                    "type": response.json()["type"]["name"],
                    "power": response.json()["power"],
                    "accuracy": response.json()["accuracy"],
                    "pp": response.json()["pp"],
                    "damage_class": response.json()["damage_class"]["name"],
                    "effect": short_effect,
                    "effect_chance": response.json()["effect_chance"],
                    "stat_changes_value": stat_changes_value,
                    "stat_changes_stat": stat_changes_stat,
                    "jpname": jpname,
                }
                print(f"追加：ID {moveid} - {move_dict[str(moveid)]['name']} except meta")
        else:
            print(f"スキップ：ID {moveid}")

        # save
    with open(filename, "w", encoding="utf-8") as f:
        f.write(json.dumps(move_dict, indent=4))
    print(f"finish {pokemon_dict[id]["name"]}")


スキップ：ID 14
スキップ：ID 34
スキップ：ID 38
スキップ：ID 46
スキップ：ID 63
スキップ：ID 73
スキップ：ID 74
スキップ：ID 76
スキップ：ID 77
スキップ：ID 79
スキップ：ID 80
スキップ：ID 89
スキップ：ID 92
スキップ：ID 113
スキップ：ID 133
スキップ：ID 156
スキップ：ID 164
スキップ：ID 173
スキップ：ID 174
スキップ：ID 182
スキップ：ID 184
スキップ：ID 188
スキップ：ID 200
スキップ：ID 202
スキップ：ID 203
スキップ：ID 204
スキップ：ID 214
スキップ：ID 230
スキップ：ID 235
スキップ：ID 241
スキップ：ID 263
スキップ：ID 270
スキップ：ID 275
スキップ：ID 282
スキップ：ID 311
スキップ：ID 331
スキップ：ID 338
スキップ：ID 388
スキップ：ID 398
スキップ：ID 402
スキップ：ID 412
スキップ：ID 414
スキップ：ID 416
スキップ：ID 437
スキップ：ID 438
スキップ：ID 447
スキップ：ID 474
スキップ：ID 482
スキップ：ID 491
スキップ：ID 496
スキップ：ID 523
スキップ：ID 572
スキップ：ID 580
スキップ：ID 707
スキップ：ID 803
スキップ：ID 805
スキップ：ID 885
finish venusaur
スキップ：ID 7
スキップ：ID 9
スキップ：ID 14
スキップ：ID 19
スキップ：ID 25
スキップ：ID 34
スキップ：ID 38
スキップ：ID 44
スキップ：ID 46
スキップ：ID 53
スキップ：ID 63
スキップ：ID 68
スキップ：ID 76
スキップ：ID 83
スキップ：ID 89
スキップ：ID 91
スキップ：ID 126
スキップ：ID 156
スキップ：ID 157
スキップ：ID 164
スキップ：ID 173
スキップ：ID 182
スキップ：ID 184
スキップ：ID 187
スキップ：ID 200
スキップ：ID 201
スキップ：ID 203
スキップ：ID

In [3]:
filename = "JSON/move.json"
with open(filename, "r", encoding="utf-8") as f:
    move_dict = json.load(f)
    move_ids = ()
    for id in move_dict:
        move = move_dict[id]
        move_ids += (move["id"],)
    move_ids = list(set(move_ids))
filename = "JSON/moveindex.json"
with open(filename, "w", encoding="utf-8") as f:
    f.write(json.dumps(move_ids))

In [4]:
# sorting
filename = "JSON/ability.json"
with open(filename, "r", encoding="utf-8") as f:
    move_dict = json.load(f)
move_dict = dict(sorted(move_dict.items(), key=lambda item: int(item[0])))
with open(filename, "w", encoding="utf-8") as f:
    f.write(json.dumps(move_dict, indent=4, ensure_ascii=False))

In [ ]:

# adding weight
filename = "pokemon.json"
with open(filename, "r", encoding="utf-8") as f:
    pokemon_dict = json.load(f)
for id in pokemon_dict:
    id = int(id)
    url = mainurl + "pokemon/" + str(id)
    response = requests.get(url)
    weight = response.json()["weight"]
    height = response.json()["height"]
    pokemon_dict[str(id)]["weight"] = weight
    pokemon_dict[str(id)]["height"] = height
    print(f"追加：ID {id} - {pokemon_dict[str(id)]['name']}")
with open(filename, "w", encoding="utf-8") as f:
    f.write(json.dumps(pokemon_dict, indent=4, ensure_ascii=False))

In [ ]:
# making ability.json
filename = "pokemon.json"
with open(filename, "r", encoding="utf-8") as f:
    pokemon_dict = json.load(f)

filename = "ability.json"
with open(filename, "r", encoding="utf-8") as f:
    try:
        ability_dict = json.load(f)
    except json.JSONDecodeError:
        ability_dict = {}
for id in pokemon_dict:
    abilities = pokemon_dict[id]["abilities"]  
    for ability in abilities:
        id = int(ability["ability_id"])
        url = mainurl + "ability/" + str(id)
        response = requests.get(url)
        effect_entries = response.json()["effect_entries"]
        for effect_entry in effect_entries:
            if effect_entry["language"]["name"] == "en":
                effect = effect_entry["effect"]
                short_effect = effect_entry["short_effect"]
                break
        natural_name = response.json()["name"]
        names = response.json()["names"]
        for name in names:
            if name["language"]["name"] == "ja-hrkt":
                jpname = name["name"]
                break
            elif name["language"]["name"] == "ja":
                jpname = name["name"]
                break
            else:
                jpname = ""
        ability_dict[id] = {
            "effect": effect,
            "short_effect": short_effect,
            "name": natural_name,
            "jpname": jpname
        }
        print(f"add {jpname},{natural_name}{effect}")
with open(filename, "w", encoding="utf-8") as f:
    f.write(json.dumps(ability_dict, indent=4, ensure_ascii=False))

In [ ]:
import json
from typing import Any

import requests

# 指定していただいたエンドポイントURL
POKEAPI_GRAPHQL_URL = "https://graphql.pokeapi.co/v1beta2"

# IDsを配列で受け取れるようにしたクエリ
FLATTEN_MOVES_QUERY = """
query getMovesByIds($ids: [Int!]) {
  move(where: {id: {_in: $ids}}) {
    id
    name
    movenames(where: {language_id: {_eq: 11}}) {
      name
    }
    type_id
    move_damage_class_id
    pp
    power
    accuracy
    movemeta {
      crit_rate
      healing
      drain
      move_meta_category_id
      move_meta_ailment_id
      ailment_chance
      stat_chance
      flinch_chance
      max_hits
      max_turns
      min_hits
      min_turns
    }
    movemetastatchanges {
      stat_id
      change
    }
    move_effect_chance
    move_effect_id
    move_target_id
    moveeffect {
      moveeffecteffecttexts(where: {language_id: {_eq: 9}}) {
        effect
      }
    }
  }
}
"""


def fetch_and_flatten_moves_json(move_ids: list[int], save_file_path: str) -> None:
    """
    指定した技IDのリストを取得し、ネストを分解してフラットなJSONとして保存する
    """
    print(f"技ID {move_ids} のデータを取得中...")

    # 1. GraphQLにリクエストを投げる
    response = requests.post(
        POKEAPI_GRAPHQL_URL,
        json={"query": FLATTEN_MOVES_QUERY, "variables": {"ids": move_ids}},
        timeout=30,
    )
    response.raise_for_status()
    print(response.json())  # デバッグ用にレスポンスを表示
    # 取得した生の配列データ
    raw_moves: list[dict[str, Any]] = response.json()["data"]["move"]

    flattened_moves = []

    # 2. 階層を分解して1階層の dict に平たくする
    for raw in raw_moves:
        # --- 深い階層から必要なデータを引っ張り出す ---

        # 日本語名
        movenames = raw.get("movenames") or []
        jpname = movenames[0]["name"] if movenames else raw["name"]

        # メタ情報
        meta_list = raw.get("movemeta") or []
        meta = meta_list[0] if meta_list else {}

        # 能力変化を {stat_id: change} のシンプルな辞書に変換
        statchanges = raw.get("movemetastatchanges") or []
        stat_changes_dict = {
            s["stat_id"]: s["change"]
            for s in statchanges
            if s.get("stat_id") is not None
        }

        # エフェクト説明文
        effect_node = raw.get("moveeffect") or {}
        effect_texts = effect_node.get("moveeffecteffecttexts") or []
        effect_desc = effect_texts[0].get("effect", "") if effect_texts else ""

        # --- すべてのデータを第1階層に並べたペッタンコの辞書を作る ---
        flat_move = {
            "id": raw["id"],
            "name": raw["name"],
            "jpname": jpname,
            "type_id": raw["type_id"],
            "damage_class_id": raw["move_damage_class_id"],
            "pp": raw["pp"],
            "power": raw["power"],
            "accuracy": raw["accuracy"],
            # メタ情報（nullの可能性があるものは default を設定）
            "target_id": raw.get("move_target_id"),
            "category_id": meta.get("move_meta_category_id", 0),
            "ailment_id": meta.get("move_meta_ailment_id", 0),
            "ailment_chance": meta.get("ailment_chance", 0),
            "crit_rate": meta.get("crit_rate", 0),
            "healing": meta.get("healing", 0),
            "drain": meta.get("drain", 0),
            "flinch_chance": meta.get("flinch_chance", 0),
            # ターン・ヒット数
            "min_hits": meta.get("min_hits"),
            "max_hits": meta.get("max_hits"),
            "min_turns": meta.get("min_turns"),
            "max_turns": meta.get("max_turns"),
            # ステータス変化の辞書
            "stat_changes": stat_changes_dict,
            "stat_chance": meta.get("stat_chance", 0),
            # ターゲットとエフェクト
            "effect_chance": raw.get("move_effect_chance"),
            "effect_id": raw.get("move_effect_id"),
            "effect_docs": effect_desc,
        }

        flattened_moves.append(flat_move)

    # 3. フラットになった辞書リストを JSON としてファイルに書き出す
    with open(save_file_path, "w", encoding="utf-8") as f:
        json.dump(flattened_moves, f, ensure_ascii=False, indent=2)

    print(
        f"✅ {len(flattened_moves)} 件の技データをペッタンコにして '{save_file_path}' に保存しました！"
    )


# ==============================
# 実行部分（テスト）
# ==============================
if __name__ == "__main__":
    # ほしい技のIDリスト（1: はたく, 2: からてチョップ）
    idsfile = "JSON/moveindex.json"
    with open(idsfile, "r", encoding="utf-8") as f:
        target_ids = json.load(f)

    # 実行してJSONファイルを作るだけ！
    fetch_and_flatten_moves_json(target_ids, "JSON/flattened_moves.json")


In [1]:
import json

import requests

POKEAPI_GRAPHQL_URL = "https://graphql.pokeapi.co/v1beta2"

# 属性データ(moveattributemaps)を追加したGraphQLクエリ
FLATTEN_MOVES_QUERY = """
query getMovesByIds($ids: [Int!]) {
  move(where: {id: {_in: $ids}}) {
    id
    name
    movenames(where: {language_id: {_eq: 11}}) {
      name
    }
    type_id
    move_damage_class_id
    pp
    power
    accuracy
    moveattributemaps {
      move_attribute_id
    }
    movemeta {
      crit_rate
      healing
      drain
      move_meta_category_id
      move_meta_ailment_id
      ailment_chance
      stat_chance
      flinch_chance
      max_hits
      max_turns
      min_hits
      min_turns
    }
    movemetastatchanges {
      stat_id
      change
    }
    move_effect_chance
    move_effect_id
    move_target_id
    moveeffect {
      moveeffecteffecttexts(where: {language_id: {_eq: 9}}) {
        effect
      }
    }
  }
}
"""


def fetch_and_flatten_moves_json(move_ids: list[int], save_file_path: str) -> None:
    """
    指定した技IDのリストを取得し、属性IDリストも含めて
    技IDをキーとした辞書（dict）として保存する
    """
    print(f"技ID {len(move_ids)} 件のデータを取得中...")

    response = requests.post(
        POKEAPI_GRAPHQL_URL,
        json={"query": FLATTEN_MOVES_QUERY, "variables": {"ids": move_ids}},
        timeout=30,
    )
    response.raise_for_status()

    raw_moves: list[dict[str, Any]] = response.json()["data"]["move"]

    # 1. 保存用のデータをリストから「辞書 (dict)」に変更
    flattened_moves_dict: dict[str, dict[str, Any]] = {}

    for raw in raw_moves:
        # 日本語名
        movenames = raw.get("movenames") or []
        jpname = movenames[0]["name"] if movenames else raw["name"]

        # 属性IDリスト [1, 4, 7] の形に抽出
        attr_maps = raw.get("moveattributemaps") or []
        attribute_ids = [
            item["move_attribute_id"]
            for item in attr_maps
            if item.get("move_attribute_id") is not None
        ]

        # メタ情報
        meta_list = raw.get("movemeta") or []
        meta = meta_list[0] if meta_list else {}

        # 能力変化
        statchanges = raw.get("movemetastatchanges") or []
        stat_changes_dict = {
            s["stat_id"]: s["change"]
            for s in statchanges
            if s.get("stat_id") is not None
        }

        # エフェクト説明文
        effect_node = raw.get("moveeffect") or {}
        effect_texts = effect_node.get("moveeffecteffecttexts") or []
        effect_desc = effect_texts[0].get("effect", "") if effect_texts else ""

        # フラットなデータオブジェクトを作成
        flat_move = {
            "id": raw["id"],
            "name": raw["name"],
            "jpname": jpname,
            "type_id": raw["type_id"],
            "damage_class_id": raw["move_damage_class_id"],
            "pp": raw["pp"],
            "power": raw["power"],
            "accuracy": raw["accuracy"],
            "attribute_ids": attribute_ids,  # 取得した属性IDリストを追加
            "target_id": raw.get("move_target_id"),
            "category_id": meta.get("move_meta_category_id", 0),
            "ailment_id": meta.get("move_meta_ailment_id", 0),
            "ailment_chance": meta.get("ailment_chance", 0),
            "crit_rate": meta.get("crit_rate", 0),
            "healing": meta.get("healing", 0),
            "drain": meta.get("drain", 0),
            "flinch_chance": meta.get("flinch_chance", 0),
            "min_hits": meta.get("min_hits"),
            "max_hits": meta.get("max_hits"),
            "min_turns": meta.get("min_turns"),
            "max_turns": meta.get("max_turns"),
            "stat_changes": stat_changes_dict,
            "stat_chance": meta.get("stat_chance", 0),
            "effect_chance": raw.get("move_effect_chance"),
            "effect_id": raw.get("move_effect_id"),
            "effect_docs": effect_desc,
        }

        # 2. 技の id を文字列キーにして辞書に格納
        move_id_str = str(raw["id"])
        flattened_moves_dict[move_id_str] = flat_move

    # 3. 辞書形式のまま JSON としてファイルに書き出す
    with open(save_file_path, "w", encoding="utf-8") as f:
        json.dump(flattened_moves_dict, f, ensure_ascii=False, indent=2)

    print(
        f"✅ {len(flattened_moves_dict)} 件の技データを 'id' キーの辞書形式で '{save_file_path}' に保存しました！"
    )


# ==============================
# 実行部分
# ==============================
if __name__ == "__main__":
    idsfile = "JSON/moveindex.json"
    with open(idsfile, "r", encoding="utf-8") as f:
        target_ids = json.load(f)

    fetch_and_flatten_moves_json(target_ids, "JSON/move_data.json")

技ID 665 件のデータを取得中...
✅ 665 件の技データを 'id' キーの辞書形式で 'JSON/move_data.json' に保存しました！
